# Ames Housing EDA Notebook

Notebook này bám sát workflow chuẩn:
1. Load train/test từ SQL (fallback CSV)
2. Đọc `data_description.txt` và nhóm biến theo domain
3. **Feature type classification** (phân loại theo bản chất thống kê/ML)
4. Missing summary before
5. Apply missing strategy theo ngữ nghĩa Ames
6. Missing summary after
7. Outlier detection + bằng chứng
8. Sensitivity analysis (with/without outlier)
9. EDA core plots (before/after log, scatter, boxplot)
10. Xuất bảng/ảnh và viết kết luận ngắn
        

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
        

## 1) Cấu hình I/O và load dữ liệu

- Nếu có `AMES_DB_URL` thì đọc từ SQL (`ames_train`, `ames_test`)
- Nếu không có thì fallback về CSV trong `data/raw`
        

In [ ]:
# Toggle
CLEAN_OLD_REPORTS = True

# Paths
BASE_DIR = Path('.')
TRAIN_CSV = BASE_DIR / 'data' / 'raw' / 'train.csv'
TEST_CSV = BASE_DIR / 'data' / 'raw' / 'test.csv'
DATA_DESCRIPTION_PATH = BASE_DIR / 'data' / 'raw' / 'data_description.txt'

REPORTS_DIR = BASE_DIR / 'reports'
FIG_DIR = REPORTS_DIR / 'figures'
TABLE_DIR = REPORTS_DIR / 'tables'

if CLEAN_OLD_REPORTS and REPORTS_DIR.exists():
    shutil.rmtree(REPORTS_DIR)

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DB_URL = os.getenv('AMES_DB_URL')
TRAIN_TABLE = 'ames_train'
TEST_TABLE = 'ames_test'

if DB_URL:
    try:
        engine = create_engine(DB_URL)
        train = pd.read_sql(f'SELECT * FROM {TRAIN_TABLE}', engine)
        test = pd.read_sql(f'SELECT * FROM {TEST_TABLE}', engine)
        data_source = f'SQL ({TRAIN_TABLE}, {TEST_TABLE})'
    except Exception as e:
        print(f'[WARN] SQL load failed: {e}')
        train = pd.read_csv(TRAIN_CSV)
        test = pd.read_csv(TEST_CSV)
        data_source = 'CSV fallback'
else:
    train = pd.read_csv(TRAIN_CSV)
    test = pd.read_csv(TEST_CSV)
    data_source = 'CSV'

print('Data source:', data_source)
print('Train shape:', train.shape)
print('Test shape:', test.shape)
train.head(3)
        

## 3) Feature Type Classification

Phân loại biến theo **bản chất dữ liệu thống kê / ML**, khác với phân nhóm theo domain knowledge ở trên.

Đây là bước quyết định:
- **Encoding** gì (Ordinal / One-Hot / Label)
- **Scaling** gì (StandardScaler / MinMax / không cần)
- **Correlation** nào phù hợp (Pearson / Spearman / Cramér's V)

Tham khảo: [Ames Data Documentation (JSE)](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt)
        

In [ ]:
# ==========================================
# FEATURE TYPE CLASSIFICATION
# ==========================================

feature_types = {

    "Numerical Continuous": [
        "LotArea",
        "GrLivArea",
        "TotalBsmtSF",
        "GarageArea",
        "MasVnrArea",
        "LotFrontage"
    ],

    "Numerical Count": [
        "FullBath",
        "HalfBath",
        "BedroomAbvGr",
        "TotRmsAbvGrd",
        "GarageCars",
        "Fireplaces"
    ],

    "Ordinal Categorical": [
        "ExterQual",
        "ExterCond",
        "KitchenQual",
        "HeatingQC",
        "BsmtQual",
        "BsmtCond",
        "GarageQual",
        "GarageCond",
        "FireplaceQu"
    ],

    "Nominal Categorical": [
        "Neighborhood",
        "MSZoning",
        "BldgType",
        "HouseStyle",
        "RoofStyle",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "SaleCondition"
    ],

    "Binary": [
        "CentralAir",
        "Street"
    ],

    "Special Coded Category": [
        "MSSubClass",
        "MoSold",
        "YrSold"
    ],

    "Target": [
        "SalePrice"
    ],

    "ID": [
        "Id"
    ]
}

for k, v in feature_types.items():
    print(f"{k}: {len(v)} features")
        

In [ ]:
"""
Tại sao phải phân loại feature?

1. Numerical Continuous
→ dùng correlation Pearson, scaling (StandardScaler), regression

2. Numerical Count
→ là số đếm, đôi khi không chuẩn hóa mạnh như continuous

3. Ordinal Categorical
→ có thứ bậc (Ex > Gd > TA > Fa > Po)
→ dùng Ordinal Encoding

4. Nominal Categorical
→ không có thứ bậc
→ dùng One-Hot Encoding

5. Binary
→ chỉ có 2 trạng thái
→ map Yes/No -> 1/0

6. Special Coded Category
→ nhìn giống số nhưng thực chất là category

Ví dụ quan trọng nhất: MSSubClass
- Giá trị: 20, 30, 40, 50, 60, ..., 190
- Ý nghĩa: mã loại nhà (dwelling type)
- KHÔNG phải số liên tục!

Nếu xử lý sai thành số:
  model sẽ hiểu 120 > 20
  và tạo quan hệ toán học giả (spurious relationship).

Tương tự: MoSold (tháng bán), YrSold (năm bán)
cũng là category chứ không nên dùng như continuous.
"""
        

In [ ]:
# ==========================================
# BẢNG TÓM TẮT: Tên biến → Bản chất → Cách xử lý
# ==========================================

processing_guide = {
    "Numerical Continuous": "Pearson correlation, StandardScaler, dùng trực tiếp cho regression",
    "Numerical Count":     "Spearman correlation, có thể scaling nhẹ hoặc giữ nguyên",
    "Ordinal Categorical": "OrdinalEncoder (Ex=5, Gd=4, TA=3, Fa=2, Po=1, None=0)",
    "Nominal Categorical": "One-Hot Encoding (pd.get_dummies / OneHotEncoder)",
    "Binary":              "LabelEncoder hoặc map thủ công (Y=1, N=0; Pave=1, Grvl=0)",
    "Special Coded Category": "Chuyển sang str rồi One-Hot Encoding (KHÔNG dùng như số)",
    "Target":              "Log1p transform để giảm skewness",
    "ID":                  "Loại bỏ khỏi tập đặc trưng (drop trước khi train)",
}

rows = []
for ftype, features in feature_types.items():
    guide = processing_guide.get(ftype, '')
    for feat in features:
        in_train = feat in train.columns if 'train' in dir() else ''
        rows.append({
            'Feature': feat,
            'Type': ftype,
            'Processing': guide,
            'InTrain': in_train
        })

feature_type_df = pd.DataFrame(rows)
feature_type_df.to_csv(TABLE_DIR / 'feature_type_classification.csv', index=False)

print(f'Tổng số features được phân loại: {len(feature_type_df)}')
print()
feature_type_df
        

## 3) Feature Type Classification

Phân loại biến theo **bản chất dữ liệu thống kê / ML**, khác với phân nhóm theo domain knowledge ở trên.

Đây là bước quyết định:
- **Encoding** gì (Ordinal / One-Hot / Label)
- **Scaling** gì (StandardScaler / MinMax / không cần)
- **Correlation** nào phù hợp (Pearson / Spearman / Cramér's V)

Tham khảo: [Ames Data Documentation (JSE)](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt)
        

In [ ]:
# ==========================================
# FEATURE TYPE CLASSIFICATION
# ==========================================

feature_types = {

    "Numerical Continuous": [
        "LotArea",
        "GrLivArea",
        "TotalBsmtSF",
        "GarageArea",
        "MasVnrArea",
        "LotFrontage"
    ],

    "Numerical Count": [
        "FullBath",
        "HalfBath",
        "BedroomAbvGr",
        "TotRmsAbvGrd",
        "GarageCars",
        "Fireplaces"
    ],

    "Ordinal Categorical": [
        "ExterQual",
        "ExterCond",
        "KitchenQual",
        "HeatingQC",
        "BsmtQual",
        "BsmtCond",
        "GarageQual",
        "GarageCond",
        "FireplaceQu"
    ],

    "Nominal Categorical": [
        "Neighborhood",
        "MSZoning",
        "BldgType",
        "HouseStyle",
        "RoofStyle",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "SaleCondition"
    ],

    "Binary": [
        "CentralAir",
        "Street"
    ],

    "Special Coded Category": [
        "MSSubClass",
        "MoSold",
        "YrSold"
    ],

    "Target": [
        "SalePrice"
    ],

    "ID": [
        "Id"
    ]
}

for k, v in feature_types.items():
    print(f"{k}: {len(v)} features")
        

In [ ]:
"""
Tại sao phải phân loại feature?

1. Numerical Continuous
→ dùng correlation Pearson, scaling (StandardScaler), regression

2. Numerical Count
→ là số đếm, đôi khi không chuẩn hóa mạnh như continuous

3. Ordinal Categorical
→ có thứ bậc (Ex > Gd > TA > Fa > Po)
→ dùng Ordinal Encoding

4. Nominal Categorical
→ không có thứ bậc
→ dùng One-Hot Encoding

5. Binary
→ chỉ có 2 trạng thái
→ map Yes/No -> 1/0

6. Special Coded Category
→ nhìn giống số nhưng thực chất là category

Ví dụ quan trọng nhất: MSSubClass
- Giá trị: 20, 30, 40, 50, 60, ..., 190
- Ý nghĩa: mã loại nhà (dwelling type)
- KHÔNG phải số liên tục!

Nếu xử lý sai thành số:
  model sẽ hiểu 120 > 20
  và tạo quan hệ toán học giả (spurious relationship).

Tương tự: MoSold (tháng bán), YrSold (năm bán)
cũng là category chứ không nên dùng như continuous.
"""
        

In [ ]:
# ==========================================
# BẢNG TÓM TẮT: Tên biến → Bản chất → Cách xử lý
# ==========================================

processing_guide = {
    "Numerical Continuous": "Pearson correlation, StandardScaler, dùng trực tiếp cho regression",
    "Numerical Count":     "Spearman correlation, có thể scaling nhẹ hoặc giữ nguyên",
    "Ordinal Categorical": "OrdinalEncoder (Ex=5, Gd=4, TA=3, Fa=2, Po=1, None=0)",
    "Nominal Categorical": "One-Hot Encoding (pd.get_dummies / OneHotEncoder)",
    "Binary":              "LabelEncoder hoặc map thủ công (Y=1, N=0; Pave=1, Grvl=0)",
    "Special Coded Category": "Chuyển sang str rồi One-Hot Encoding (KHÔNG dùng như số)",
    "Target":              "Log1p transform để giảm skewness",
    "ID":                  "Loại bỏ khỏi tập đặc trưng (drop trước khi train)",
}

rows = []
for ftype, features in feature_types.items():
    guide = processing_guide.get(ftype, '')
    for feat in features:
        in_train = feat in train.columns if 'train' in dir() else ''
        rows.append({
            'Feature': feat,
            'Type': ftype,
            'Processing': guide,
            'InTrain': in_train
        })

feature_type_df = pd.DataFrame(rows)
feature_type_df.to_csv(TABLE_DIR / 'feature_type_classification.csv', index=False)

print(f'Tổng số features được phân loại: {len(feature_type_df)}')
print()
feature_type_df
        

## 2) Domain knowledge từ `data_description.txt`

- Chia nhóm biến theo business logic
- Tạo bảng `feature_groups.csv`
        

In [ ]:
if DATA_DESCRIPTION_PATH.exists():
    desc_text = DATA_DESCRIPTION_PATH.read_text(encoding='utf-8', errors='ignore')
    print('Loaded data_description.txt, length:', len(desc_text))
    print(desc_text[:1200])
else:
    desc_text = ''
    print('[WARN] data_description.txt not found')

FEATURE_GROUPS = {
    'size_area': ['LotArea', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF'],
    'quality_condition': ['OverallQual', 'OverallCond', 'KitchenQual', 'ExterQual', 'HeatingQC'],
    'house_age': ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt'],
    'location': ['Neighborhood', 'Condition1', 'Street'],
    'utilities': ['GarageCars', 'GarageArea', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'PoolArea'],
    'transaction': ['SaleType', 'SaleCondition', 'YrSold', 'MoSold'],
}

fg_rows = []
for grp, cols in FEATURE_GROUPS.items():
    for col in cols:
        fg_rows.append({'Group': grp, 'Feature': col, 'InTrainColumns': col in train.columns})
feature_groups_df = pd.DataFrame(fg_rows)
feature_groups_df.to_csv(TABLE_DIR / 'feature_groups.csv', index=False)
feature_groups_df.head(20)
        

## 3) Feature Type Classification

Phân loại biến theo **bản chất dữ liệu thống kê / ML**, khác với phân nhóm theo domain knowledge ở trên.

Đây là bước quyết định:
- **Encoding** gì (Ordinal / One-Hot / Label)
- **Scaling** gì (StandardScaler / MinMax / không cần)
- **Correlation** nào phù hợp (Pearson / Spearman / Cramér's V)

Tham khảo: [Ames Data Documentation (JSE)](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt)
        

In [ ]:
# ==========================================
# FEATURE TYPE CLASSIFICATION
# ==========================================

feature_types = {

    "Numerical Continuous": [
        "LotArea",
        "GrLivArea",
        "TotalBsmtSF",
        "GarageArea",
        "MasVnrArea",
        "LotFrontage"
    ],

    "Numerical Count": [
        "FullBath",
        "HalfBath",
        "BedroomAbvGr",
        "TotRmsAbvGrd",
        "GarageCars",
        "Fireplaces"
    ],

    "Ordinal Categorical": [
        "ExterQual",
        "ExterCond",
        "KitchenQual",
        "HeatingQC",
        "BsmtQual",
        "BsmtCond",
        "GarageQual",
        "GarageCond",
        "FireplaceQu"
    ],

    "Nominal Categorical": [
        "Neighborhood",
        "MSZoning",
        "BldgType",
        "HouseStyle",
        "RoofStyle",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "SaleCondition"
    ],

    "Binary": [
        "CentralAir",
        "Street"
    ],

    "Special Coded Category": [
        "MSSubClass",
        "MoSold",
        "YrSold"
    ],

    "Target": [
        "SalePrice"
    ],

    "ID": [
        "Id"
    ]
}

for k, v in feature_types.items():
    print(f"{k}: {len(v)} features")
        

In [ ]:
"""
Tại sao phải phân loại feature?

1. Numerical Continuous
→ dùng correlation Pearson, scaling (StandardScaler), regression

2. Numerical Count
→ là số đếm, đôi khi không chuẩn hóa mạnh như continuous

3. Ordinal Categorical
→ có thứ bậc (Ex > Gd > TA > Fa > Po)
→ dùng Ordinal Encoding

4. Nominal Categorical
→ không có thứ bậc
→ dùng One-Hot Encoding

5. Binary
→ chỉ có 2 trạng thái
→ map Yes/No -> 1/0

6. Special Coded Category
→ nhìn giống số nhưng thực chất là category

Ví dụ quan trọng nhất: MSSubClass
- Giá trị: 20, 30, 40, 50, 60, ..., 190
- Ý nghĩa: mã loại nhà (dwelling type)
- KHÔNG phải số liên tục!

Nếu xử lý sai thành số:
  model sẽ hiểu 120 > 20
  và tạo quan hệ toán học giả (spurious relationship).

Tương tự: MoSold (tháng bán), YrSold (năm bán)
cũng là category chứ không nên dùng như continuous.
"""
        

In [ ]:
# ==========================================
# BẢNG TÓM TẮT: Tên biến → Bản chất → Cách xử lý
# ==========================================

processing_guide = {
    "Numerical Continuous": "Pearson correlation, StandardScaler, dùng trực tiếp cho regression",
    "Numerical Count":     "Spearman correlation, có thể scaling nhẹ hoặc giữ nguyên",
    "Ordinal Categorical": "OrdinalEncoder (Ex=5, Gd=4, TA=3, Fa=2, Po=1, None=0)",
    "Nominal Categorical": "One-Hot Encoding (pd.get_dummies / OneHotEncoder)",
    "Binary":              "LabelEncoder hoặc map thủ công (Y=1, N=0; Pave=1, Grvl=0)",
    "Special Coded Category": "Chuyển sang str rồi One-Hot Encoding (KHÔNG dùng như số)",
    "Target":              "Log1p transform để giảm skewness",
    "ID":                  "Loại bỏ khỏi tập đặc trưng (drop trước khi train)",
}

rows = []
for ftype, features in feature_types.items():
    guide = processing_guide.get(ftype, '')
    for feat in features:
        in_train = feat in train.columns if 'train' in dir() else ''
        rows.append({
            'Feature': feat,
            'Type': ftype,
            'Processing': guide,
            'InTrain': in_train
        })

feature_type_df = pd.DataFrame(rows)
feature_type_df.to_csv(TABLE_DIR / 'feature_type_classification.csv', index=False)

print(f'Tổng số features được phân loại: {len(feature_type_df)}')
print()
feature_type_df
        

## 4) Missing values before cleaning
        

In [ ]:
def missing_summary(df):
    s = df.isna().sum()
    s = s[s > 0].sort_values(ascending=False)
    return pd.DataFrame({
        'Column': s.index,
        'MissingCount': s.values,
        'MissingPercent': (s.values / len(df) * 100).round(2)
    })

missing_before_train = missing_summary(train)
missing_before_test = missing_summary(test)
missing_before_train.to_csv(TABLE_DIR / 'missing_summary_before.csv', index=False)
missing_before_train.to_csv(TABLE_DIR / 'missing_train_before.csv', index=False)
missing_before_test.to_csv(TABLE_DIR / 'missing_test_before.csv', index=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=missing_before_train.head(30), x='Column', y='MissingPercent')
plt.xticks(rotation=90)
plt.title('Missing Value Percentage (Train, Before)')
plt.tight_layout()
plt.savefig(FIG_DIR / '01_missing_train_before.png', dpi=150)
plt.close()

missing_before_train.head(20)
        

## 5) Apply missing strategy theo ngữ nghĩa Ames

- `None`: structural absence (không có feature)
- `0`: structural numeric absence
- median/mode: true missing
- median/mode lấy từ **train** rồi áp sang test
        

In [ ]:
NO_FEATURE_COLS = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
]

ZERO_FILL_COLS = [
    'GarageCars', 'GarageArea', 'BsmtFinSF1', 'BsmtFinSF2',
    'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath'
]

TRUE_NUMERIC_MISSING = ['LotFrontage']
TRUE_CATEGORICAL_MISSING = ['Electrical']


def safe_mode(series, default='None'):
    m = series.mode(dropna=True)
    return m.iloc[0] if len(m) > 0 else default


def apply_missing_strategy(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()
    strategy_rows = []

    for col in NO_FEATURE_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna('None')
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna('None')
        if col in train_df.columns or col in test_df.columns:
            strategy_rows.append({'Column': col, 'Strategy': "fillna('None')", 'Reason': 'structural absence'})

    for col in ZERO_FILL_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(0)
        if col in train_df.columns or col in test_df.columns:
            strategy_rows.append({'Column': col, 'Strategy': 'fillna(0)', 'Reason': 'structural numeric absence'})

    for col in TRUE_NUMERIC_MISSING:
        if col in train_df.columns:
            med = train_df[col].median()
            train_df[col] = train_df[col].fillna(med)
            if col in test_df.columns:
                test_df[col] = test_df[col].fillna(med)
            strategy_rows.append({'Column': col, 'Strategy': f'fillna(train_median={med:.4f})', 'Reason': 'true numeric missing'})

    for col in TRUE_CATEGORICAL_MISSING:
        if col in train_df.columns:
            md = safe_mode(train_df[col], default='SBrkr')
            train_df[col] = train_df[col].fillna(md)
            if col in test_df.columns:
                test_df[col] = test_df[col].fillna(md)
            strategy_rows.append({'Column': col, 'Strategy': f'fillna(train_mode={md})', 'Reason': 'true categorical missing'})

    # Fallback for remaining missing
    for col in train_df.columns:
        if train_df[col].isna().any():
            if pd.api.types.is_numeric_dtype(train_df[col]):
                med = train_df[col].median()
                fill_val = 0.0 if pd.isna(med) else med
                train_df[col] = train_df[col].fillna(fill_val)
                if col in test_df.columns:
                    test_df[col] = test_df[col].fillna(fill_val)
                strategy_rows.append({'Column': col, 'Strategy': f'fallback median={fill_val:.4f}', 'Reason': 'remaining numeric missing'})
            else:
                md = safe_mode(train_df[col], default='None')
                train_df[col] = train_df[col].fillna(md)
                if col in test_df.columns:
                    test_df[col] = test_df[col].fillna(md)
                strategy_rows.append({'Column': col, 'Strategy': f'fallback mode={md}', 'Reason': 'remaining categorical missing'})

    # If test still has any missing for test-only columns
    for col in test_df.columns:
        if test_df[col].isna().any():
            if pd.api.types.is_numeric_dtype(test_df[col]):
                med = test_df[col].median()
                fill_val = 0.0 if pd.isna(med) else med
                test_df[col] = test_df[col].fillna(fill_val)
            else:
                md = safe_mode(test_df[col], default='None')
                test_df[col] = test_df[col].fillna(md)

    strategy_df = pd.DataFrame(strategy_rows).drop_duplicates(subset=['Column', 'Strategy'])
    return train_df, test_df, strategy_df


train_clean, test_clean, missing_strategy_df = apply_missing_strategy(train, test)
missing_strategy_df.to_csv(TABLE_DIR / 'missing_strategy_applied.csv', index=False)

missing_after_train = missing_summary(train_clean)
missing_after_test = missing_summary(test_clean)
missing_after_train.to_csv(TABLE_DIR / 'missing_summary_after.csv', index=False)
missing_after_train.to_csv(TABLE_DIR / 'missing_train_after.csv', index=False)
missing_after_test.to_csv(TABLE_DIR / 'missing_test_after.csv', index=False)

plt.figure(figsize=(14, 6))
if len(missing_after_train) > 0:
    sns.barplot(data=missing_after_train.head(30), x='Column', y='MissingPercent')
    plt.xticks(rotation=90)
else:
    plt.text(0.5, 0.5, 'No missing values in train after cleaning', ha='center', va='center', fontsize=14)
    plt.axis('off')
plt.title('Missing Value Percentage (Train, After)')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_missing_train_after.png', dpi=150)
plt.close()

print('Train missing after:', train_clean.isna().sum().sum())
print('Test missing after :', test_clean.isna().sum().sum())
missing_strategy_df.head(20)
        

## 6) Outlier detection + evidence + sensitivity analysis

Rule domain của Ames thường kiểm tra:
`GrLivArea > 4000` và `SalePrice < 300000`
        

In [ ]:
outlier_mask = (train_clean['GrLivArea'] > 4000) & (train_clean['SalePrice'] < 300000)
outliers = train_clean.loc[outlier_mask].copy()
outliers[['GrLivArea', 'SalePrice']].to_csv(TABLE_DIR / 'domain_outlier_candidates.csv', index=False)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=train_clean['GrLivArea'], y=train_clean['SalePrice'], alpha=0.7, label='Inlier')
if outlier_mask.any():
    sns.scatterplot(
        x=train_clean.loc[outlier_mask, 'GrLivArea'],
        y=train_clean.loc[outlier_mask, 'SalePrice'],
        color='red',
        s=80,
        label='Domain outlier'
    )
plt.title('GrLivArea vs SalePrice')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_outlier_grlivarea_saleprice.png', dpi=150)
plt.savefig(FIG_DIR / 'outlier_grlivarea_saleprice.png', dpi=150)
plt.close()

print('Outlier count:', int(outlier_mask.sum()))
outliers[['GrLivArea', 'SalePrice']]
        

In [ ]:
baseline_features = [c for c in ['OverallQual', 'GrLivArea', 'GarageCars'] if c in train_clean.columns]


def eval_baseline(df):
    X = df[baseline_features].copy()
    y = df['SalePrice'].copy()
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    return {
        'MAE': mean_absolute_error(y_valid, pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
        'R2': r2_score(y_valid, pred),
    }

metrics_with = eval_baseline(train_clean)
metrics_without = eval_baseline(train_clean.loc[~outlier_mask])

outlier_sensitivity_df = pd.DataFrame([
    {'Scenario': 'with_outliers', **metrics_with},
    {'Scenario': 'without_outliers', **metrics_without},
])
outlier_sensitivity_df.to_csv(TABLE_DIR / 'outlier_sensitivity_metrics.csv', index=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=outlier_sensitivity_df, x='Scenario', y='RMSE')
plt.title('Outlier Sensitivity Analysis (RMSE)')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_outlier_sensitivity_rmse.png', dpi=150)
plt.close()

outlier_sensitivity_df
        

## 7) Core EDA plots

- `SalePrice` before/after `log1p`
- `GrLivArea` vs `SalePrice`
- `OverallQual` vs `SalePrice`
        

In [ ]:
train_eda = train_clean.copy()
train_eda['SalePrice_log'] = np.log1p(train_eda['SalePrice'])

# before log
plt.figure(figsize=(10, 6))
sns.histplot(train_eda['SalePrice'], kde=True)
plt.title('SalePrice Distribution (Before Log)')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_saleprice_distribution.png', dpi=150)
plt.savefig(FIG_DIR / 'saleprice_before_log.png', dpi=150)
plt.close()

# after log
plt.figure(figsize=(10, 6))
sns.histplot(train_eda['SalePrice_log'], kde=True, color='teal')
plt.title('SalePrice Distribution (After Log1p)')
plt.tight_layout()
plt.savefig(FIG_DIR / '06_saleprice_log_distribution.png', dpi=150)
plt.savefig(FIG_DIR / 'saleprice_after_log.png', dpi=150)
plt.close()

# scatter GrLivArea vs SalePrice
plt.figure(figsize=(10, 6))
sns.scatterplot(x=train_eda['GrLivArea'], y=train_eda['SalePrice'], alpha=0.7)
sns.regplot(x=train_eda['GrLivArea'], y=train_eda['SalePrice'], scatter=False, ci=None, color='red')
plt.title('GrLivArea vs SalePrice')
plt.tight_layout()
plt.savefig(FIG_DIR / 'grlivarea_vs_saleprice.png', dpi=150)
plt.savefig(FIG_DIR / '07_scatter_2_GrLivArea.png', dpi=150)
plt.close()

# boxplot OverallQual vs SalePrice
plt.figure(figsize=(12, 6))
sns.boxplot(x=train_eda['OverallQual'], y=train_eda['SalePrice'])
plt.title('OverallQual vs SalePrice')
plt.tight_layout()
plt.savefig(FIG_DIR / 'overallqual_vs_saleprice.png', dpi=150)
plt.savefig(FIG_DIR / '10_segment_overallqual.png', dpi=150)
plt.close()

print('Saved core EDA plots.')
        

## 8) Lưu kết luận ngắn
        

In [ ]:
summary_lines = []
summary_lines.append('# EDA Summary')
summary_lines.append('')
summary_lines.append(f'- Data source: {data_source}')
summary_lines.append(f'- Train shape: {train_clean.shape[0]} x {train_clean.shape[1]}')
summary_lines.append(f'- Test shape: {test_clean.shape[0]} x {test_clean.shape[1]}')
summary_lines.append(f'- Domain outlier count (GrLivArea > 4000 & SalePrice < 300000): {int(outlier_mask.sum())}')
summary_lines.append('')
summary_lines.append('## Outlier sensitivity metrics')
summary_lines.append(outlier_sensitivity_df.to_string(index=False))
summary_lines.append('')
summary_lines.append('## Notes')
summary_lines.append('- Missing values are handled by semantic categories (None / 0 / train median/mode).')
summary_lines.append('- SalePrice is right-skewed; log1p transformation improves target distribution symmetry.')
summary_lines.append('- Outliers are not removed blindly; impact is verified via sensitivity analysis.')

summary_text = '\n'.join(summary_lines)
(REPORTS_DIR / 'eda_summary.md').write_text(summary_text, encoding='utf-8')

print(summary_text)
        

## 9) Checklist output

Notebook này sinh ra các file chính:
- `reports/tables/feature_groups.csv`
- `reports/tables/missing_summary_before.csv`
- `reports/tables/missing_summary_after.csv`
- `reports/tables/missing_strategy_applied.csv`
- `reports/tables/domain_outlier_candidates.csv`
- `reports/tables/outlier_sensitivity_metrics.csv`
- `reports/figures/saleprice_before_log.png`
- `reports/figures/saleprice_after_log.png`
- `reports/figures/grlivarea_vs_saleprice.png`
- `reports/figures/overallqual_vs_saleprice.png`
- `reports/eda_summary.md`
        